In [1]:
import sys
import json 
import joblib
import gc
from tqdm import tqdm
import os
from typing import List, Dict, Tuple

# TO CHANGE
BASEDIR = "../../"
sys.path.insert(0, BASEDIR)

from src.pipelines.memorize import MemPipelineConfig, MemPipeline, LLMExtractorConfig, LLMUpdatorConfig
from src.kg_model import KnowledgeGraphModel, EmbeddingsModelConfig, GraphModelConfig, EmbedderModelConfig
from src.db_drivers.graph_driver import GraphDBConnectionConfig, GraphDriverConfig
from src.db_drivers.vector_driver import VectorDBConnectionConfig, VectorDriverConfig

# gigachat key
#GIGACHAT_CREDS = 'OWUwOGUzOWEtMjJiNi00YmMxLThmMmItNzMwNjM2MTI2YmYxOjg2ODdiOTVhLTZkNDctNGFjOC1iMmViLTEyNDA5MmFiN2Q5Mw=='
# openai key
#API_KEY = "'sk-861mINAavom2SSBqgrI82D4thMOfqT37knCof2o0H0T3BlbkFJ2gdVXJuVjNesNNP2aeUwPoBpZP3a3R1gn1kqv97CsA'"

gc.collect()

/home/dzigen/Desktop/PersonalAI/.pai_venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


976

### Setting knowledge-graph configuration

In [2]:
# TO CHANGE
HYPER_PARAMS = {
    'DATASET_PATH': '../../data/qa_datasets/diaasqa/Augment_DiaASQ.json',
    'DATASET_NAME': 'diaasqa',
    'KNOWLEDGE_GRAPH_NAME': 'gigachat_full',
    'EMBEDDER_MODEL_PATH': '../../models/intfloat/multilingual-e5-small',
    'DELETE_OBSOLETE_INFO': False
}
# TO CHANGE

In [3]:
BASE_PATH = "../../data/knowledge_graphs/"
DATASET_PATH = BASE_PATH + f"{HYPER_PARAMS['DATASET_NAME']}/"
KG_PATH = DATASET_PATH + f"{HYPER_PARAMS['KNOWLEDGE_GRAPH_NAME']}/"

HYPER_PARAMS_PATH = KG_PATH + 'hyperparameters.json'
EXTRACTED_TRIPLETS_PATH = KG_PATH + "extracted_triplets"
GRAPH_DRIVER_CONFIG_PATH = KG_PATH + "graph_config"
EMBEDDINGS_DRIVER_CONFIG_PATH = KG_PATH + "embeddings_config"
MEM_PIPELINE_CONFIG_PATH = KG_PATH + "mem_pipeline_config"

VECTORIZED_DB_PATH = KG_PATH + "embeddings_part/"
GRAPH_DB_PATH = KG_PATH + "graph_part/"

In [ ]:
if not os.path.exists(BASE_PATH):
    raise ValueError(f"Директории не существует: {BASE_PATH}")
if not os.path.exists(DATASET_PATH):
    raise ValueError(f"Директории не существует: {DATASET_PATH}")
if os.path.exists(KG_PATH):
    raise ValueError(f"Директория существует: {KG_PATH}")

os.mkdir(KG_PATH)
os.mkdir(VECTORIZED_DB_PATH)
os.mkdir(GRAPH_DB_PATH)

In [4]:
print(VECTORIZED_DB_PATH)
print(GRAPH_DB_PATH)

../../data/knowledge_graphs/diaasqa/gigachat_full/embeddings_part/
../../data/knowledge_graphs/diaasqa/gigachat_full/graph_part/


In [5]:
# Setting knowledge graph

graph_config = GraphModelConfig(
    driver_config=GraphDriverConfig(
        db_vendor='neo4j', 
        db_config=GraphDBConnectionConfig(
            uri="bolt://localhost:7687", params={'user': "neo4j", 'pwd': 'password'}, 
            need_to_clear=True)))

embed_config = EmbeddingsModelConfig(
    nodesdb_driver_config=VectorDriverConfig(
        db_vendor='chroma',
        db_config=VectorDBConnectionConfig(
            path=VECTORIZED_DB_PATH, db_info={'db': 'personalaidb', 'table': "vectorized_nodes"}, need_to_clear=True)),
    tripletsdb_driver_config=VectorDriverConfig(
        db_vendor='chroma', 
        db_config=VectorDBConnectionConfig(
            path=VECTORIZED_DB_PATH, db_info={'db': 'personalaidb', 'table': "vectorized_triplets"}, need_to_clear=True)),
    embedder_config=EmbedderModelConfig(model_name_or_path=HYPER_PARAMS['EMBEDDER_MODEL_PATH']))

kg_model = KnowledgeGraphModel(
    graph_config=graph_config,
    embeddings_config=embed_config)

No sentence-transformers model found with name ../../models/intfloat/multilingual-e5-small. Creating a new one with mean pooling.


In [6]:
print(kg_model.embeddings_struct.vectordbs['nodes'].count_items())
print(kg_model.embeddings_struct.vectordbs['triplets'].count_items())
print(kg_model.graph_struct.db_conn.count_items())

0
0
{'triplets': 0, 'nodes': 0}


In [7]:
# Setting Memorization Pipeline

mem_config = MemPipelineConfig(
    extractor_config=LLMExtractorConfig(),
    updator_config=LLMUpdatorConfig(
        delete_obsolete_info=HYPER_PARAMS['DELETE_OBSOLETE_INFO']))

mem_pipeline = MemPipeline(kg_model, mem_config)

### Loading dataset

In [8]:
def custom_diaasqa_load(dataset_path: str) -> List[Tuple[str, Dict[str, str]]]:
    with open(dataset_path, 'r', encoding='utf-8') as fd:
        data = json.loads(fd.read())

    data_pairs = []
    for item in data['data']:
        data_pairs.append((item['text_dialog'], {'time': item['time'].split(',')[0]}))

    return data_pairs

In [9]:
CUSTOM_LOAD_FUNCS = {
    'diaasqa': custom_diaasqa_load
}

In [10]:
dataset = CUSTOM_LOAD_FUNCS[HYPER_PARAMS['DATASET_NAME']](HYPER_PARAMS['DATASET_PATH'])

In [11]:
print(len(dataset))

3483


### Creating knowledge graph

In [12]:
saved_triplets = []
for item in tqdm(dataset):
    text, properties = item[0], item[1]
    extracted_triplets, _ = mem_pipeline.remember(text, properties)
    saved_triplets.append(extracted_triplets)

  1%|▏         | 48/3483 [07:19<8:44:14,  9.16s/it] 


KeyboardInterrupt: 

### Saving log inforamtion

In [ ]:
with open(HYPER_PARAMS_PATH, 'w', encodings='utf-8') as fd:
    fd.write(json.dumps(HYPER_PARAMS, ensure_ascii=False, indent=1))

In [ ]:
joblib.dump(saved_triplets, EXTRACTED_TRIPLETS_PATH)

joblib.dump(saved_triplets, GRAPH_DRIVER_CONFIG_PATH)
joblib.dump(saved_triplets, EMBEDDINGS_DRIVER_CONFIG_PATH)
joblib.dump(saved_triplets, MEM_PIPELINE_CONFIG_PATH)